# 00 - Preparación / compilación de bases EPH

**Este es el notebook base del proyecto.** Compila las bases de la EPH descargadas
manualmente del INDEC y subidas a **Google Drive** (carpeta `carga_EPH`): une la base de
**individuos** con la de **hogares** por el código de vínculo (`CODUSU` + `NRO_HOGAR`) y
deja los datasets listos en `data/processed/` para que **el resto de los notebooks
(01-05) los procesen** sin tener que volver a descargar ni unir las bases.

**Salidas (en `data/processed/`):**
- `eph_T<Q><YY>.parquet`: un archivo por trimestre, ya con individuos+hogares unidos.
- `eph_panel.parquet`: panel con todos los trimestres concatenados (con columnas `ANIO` y `TRIMESTRE`).

**Flujo:** Setup (clonar repo + montar Drive) → Diagnóstico de `carga_EPH` →
Trimestres disponibles → Compilar panel → Verificación (merge + quiebre de esquema 4T2023).

**Cómo agregar un trimestre nuevo:**
1. Descargar del INDEC el `.zip` del trimestre (ej. `EPH_usu_4_Trim_2025_txt.zip`),
   que contiene `usu_individual_T<Q><YY>.txt` y `usu_hogar_T<Q><YY>.txt`.
2. Subir el `.zip` (sin descomprimir) a Google Drive, carpeta `carga_EPH`.
3. Volver a correr este notebook: detecta automáticamente todos los trimestres
   presentes en `carga_EPH` (no hace falta editar ninguna lista).

> ⚠️ **Quiebre de esquema en 4T2023.** Los trimestres ≤ T3-2023 tienen menos columnas
> (177 individual / 88 hogar) que los ≥ T4-2023 (235 / 98). Al concatenar el panel, las
> variables nuevas (`EMPLEO`, `SECTOR`, ingresos/pensiones desagregados, deciles
> `P_DECCF`) quedan vacías para los trimestres viejos. Ver `.claude/memoria_EPH.md`.

## Setup (Colab)

Clona el repo para tener acceso a `src/data_loader.py`, y monta Google Drive para
leer los `.zip` subidos a `carga_EPH`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

## Diagnóstico: ¿qué hay en carga_EPH?

Si la celda "Disponibles" más abajo devuelve una lista vacía, correr esta celda
primero para ver qué archivos detecta Colab en la carpeta de Drive y qué contiene
un `.zip` de ejemplo.

In [ ]:
from src.data_loader import DRIVE_DIR
import os, zipfile

print("DRIVE_DIR:", DRIVE_DIR, "| existe:", os.path.isdir(DRIVE_DIR))

if os.path.isdir(DRIVE_DIR):
    archivos = os.listdir(DRIVE_DIR)
    print(f"Archivos encontrados ({len(archivos)}):")
    for a in archivos[:10]:
        print(" -", a)

    zips = [a for a in archivos if a.lower().endswith(".zip")]
    if zips:
        ejemplo = os.path.join(DRIVE_DIR, zips[0])
        print(f"\nContenido de {zips[0]}:")
        with zipfile.ZipFile(ejemplo) as zf:
            for n in zf.namelist():
                print(" -", n)

## Chequeo de disponibilidad

Escanea `/content/drive/MyDrive/carga_EPH` (y `data/raw/` del repo) buscando los `.zip`
del INDEC, y lista los trimestres detectados (con ambas bases, individual y hogar).

In [ ]:
from src.data_loader import list_available_quarters, _period_tag

available = list_available_quarters()
print("Disponibles:", [_period_tag(y, p) for y, p in available])

## Construcción del panel

Une individuos + hogares por trimestre (`CODUSU` + `NRO_HOGAR`), agrega `ANIO`/`TRIMESTRE`, y guarda en `data/processed/`.

In [ ]:
from src.data_loader import build_panel

panel = build_panel(available, save=True)
panel.shape

In [ ]:
# Verificación rápida del panel compilado
print("Filas:", len(panel), "| Columnas:", panel.shape[1])
print("\nTrimestres incluidos:")
print(panel.groupby(["ANIO", "TRIMESTRE"]).size())

panel[["ANIO", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "COMPONENTE"]].head()

## Verificación del quiebre de esquema (4T2023)

Las variables incorporadas desde 4T2023 (`EMPLEO`, `SECTOR`, etc.) existen como columna
en el panel pero están vacías en los trimestres viejos. Esta celda muestra desde qué
trimestre cada variable nueva tiene datos, para tenerlo presente al filtrar en los
notebooks 01-05.

In [ ]:
# Variables del esquema "nuevo" (desde 4T2023): primer trimestre con datos no nulos
vars_nuevas = ["EMPLEO", "SECTOR", "P_DECCF", "V2_01_M", "V5_01_M"]

for col in vars_nuevas:
    if col not in panel.columns:
        print(f"{col}: no está en el panel")
        continue
    con_datos = panel.loc[panel[col].notna(), ["ANIO", "TRIMESTRE"]]
    if con_datos.empty:
        print(f"{col}: sin datos en ningún trimestre")
    else:
        primer = con_datos.sort_values(["ANIO", "TRIMESTRE"]).iloc[0]
        print(f"{col}: primer trimestre con datos -> {int(primer.ANIO)}T{int(primer.TRIMESTRE)}")

## Subir resultados a GitHub (opcional)

Los archivos `.parquet` generados en `data/processed/` quedan disponibles para los demás notebooks vía `raw.githubusercontent.com` una vez que se comiteen y pusheen al repo (esto se hace desde el entorno local, no desde Colab).